# Analysis Main Notebook

이 노트북은 `Analysis` 안의 분석 도구를 한 곳에서 호출합니다. SSH 환경에서도 결과를 볼 수 있도록 그래프는 `trends`, JSON과 최종 표 CSV는 `final` 아래에 저장합니다.

## 0. 공통 설정

In [ ]:
from pathlib import Path
import json
import pandas as pd
import sys

# Determine repository root by climbing from the current working directory.
# This makes imports and relative result/output paths work from the notebook
# even if the notebook server starts from a subdirectory.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from Analysis.analyzer import Analyzer
from Analysis.distance_metrics import nearestStats
from Analysis.statistics import calcStats, printStats, reportCluster
from Analysis.result_io import finalPoints, listBands, loadAlgoRuns
from Analysis.trends import (
    plotConverge,
    saveReport,
    coverSummary,
)

RESULTS_ROOT = REPO_ROOT / "__RESULTS__"
ALGORITHMS = ["ga", "pso", "drl", "greedy"]
MAP_NAME = "gangjin.down"
SEED_BAND = None
GRID_M = 5.0
COVERAGE = 45
TARGET_VALUES = (2, 3)

ANALYSIS_ROOT = RESULTS_ROOT / "analysis"
MAP_DIR_NAME = MAP_NAME.replace("/", "_")


def normalizeAlgorithms(value: str | list[str] | tuple[str, ...]) -> list[str]:
    if isinstance(value, str):
        return [value]
    return [str(item) for item in value]


def buildContext(algorithm: str) -> dict[str, object]:
    bands = listBands(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
    )
    selected_band = SEED_BAND if SEED_BAND is not None else (bands[0] if bands else None)
    algo_map_dir = ANALYSIS_ROOT / algorithm / MAP_DIR_NAME
    trends_dir = algo_map_dir / "trends"
    final_dir = algo_map_dir / "final"
    trends_dir.mkdir(parents=True, exist_ok=True)
    final_dir.mkdir(parents=True, exist_ok=True)
    return {
        "algorithm": algorithm,
        "bands": bands,
        "selected_band": selected_band,
        "run_dir": RESULTS_ROOT / algorithm / MAP_NAME,
        "single_run_dir": RESULTS_ROOT / algorithm / MAP_NAME / selected_band
        if selected_band
        else RESULTS_ROOT / algorithm / MAP_NAME,
        "trends_dir": trends_dir,
        "final_dir": final_dir,
        "report_dir": trends_dir,
    }


ALGORITHMS = normalizeAlgorithms(ALGORITHMS)
CONTEXTS = {algorithm: buildContext(algorithm) for algorithm in ALGORITHMS}
pd.DataFrame.from_dict(CONTEXTS, orient="index")[
    ["selected_band", "trends_dir", "final_dir"]
]


## 1. 결과 파일과 seed band 확인

In [ ]:
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    bands = list(context["bands"])
    records = loadAlgoRuns(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
        seed_band=SEED_BAND,
    )
    context["records"] = records
    print(f"[{algorithm}] seed bands: {bands}")
    print(f"[{algorithm}] run count: {len(records)}")

pd.DataFrame(
    {
        algorithm: {
            "bands": len(context["bands"]),
            "runs": len(context["records"]),
            "selected_band": context["selected_band"],
        }
        for algorithm, context in CONTEXTS.items()
    }
).T


## 2. 단일 run 세대별 추이

`Analyzer`는 하나의 JSON run에 대해 센서 수, 커버리지, fitness 추이를 확인합니다.

In [ ]:
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    single_run_dir = Path(context["single_run_dir"])
    trends_dir = Path(context["trends_dir"])
    analyzer = Analyzer(result_root_path=str(single_run_dir), file_path=0)

    analyzer.plot_evolution_trend(
        xtick_step=50,
        include_corners=True,
        figsize=(10, 3),
        dpi=150,
        show=False,
        save_path=trends_dir / "sensor_count.png",
        close=True,
    )
    analyzer.plot_coverage_trend(
        xtick_step=50,
        figsize=(10, 3),
        dpi=150,
        show=False,
        save_path=trends_dir / "coverage.png",
        close=True,
    )
    analyzer.plot_fitness_trend(
        xtick_step=50,
        figsize=(10, 3),
        dpi=150,
        show=False,
        save_path=trends_dir / "fitness.png",
        close=True,
    )
    print(f"[{algorithm}] saved single-run plots to {trends_dir}")


## 3. 선택 band의 기본 통계

In [ ]:
stats_rows = []
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    single_run_dir = Path(context["single_run_dir"])
    print(f"[{algorithm}]")
    printStats(str(single_run_dir))
    stats = calcStats(str(single_run_dir))
    stats_rows.append(
        {
            "algorithm": algorithm,
            "runs": stats[0],
            "coverage_mean": stats[1],
            "coverage_std": stats[2],
            "corner_mean": stats[3],
            "total_mean": stats[5],
            "optimizer_sec_mean": stats[9],
        }
    )

stats_df = pd.DataFrame(stats_rows).set_index("algorithm")
stats_df.round(3)


## 4. 센서 간 거리 분석

In [ ]:
distance_rows = []
cluster_rows = []
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    single_run_dir = Path(context["single_run_dir"])
    records = list(context["records"])
    cluster = reportCluster(str(single_run_dir), MAP_NAME, grid_m=GRID_M)
    if cluster is not None:
        cluster["algorithm"] = algorithm
        cluster_rows.append(cluster)
    for path, run in records:
        row = nearestStats(finalPoints(run), grid_m=GRID_M)
        row["algorithm"] = algorithm
        row["run_name"] = run.get("run_name", path.stem)
        distance_rows.append(row)

distance_df = pd.DataFrame(distance_rows).set_index(["algorithm", "run_name"])
cluster_df = pd.DataFrame(cluster_rows).set_index("algorithm")
cluster_df.round(3)


## 5. 초기 seed 센서수별 수렴 분석

`metric='best'`는 세대별 best solution 센서수를 기준으로 수렴을 봅니다. `metric='avg'`로 바꾸면 로그의 평균 센서 수 기준으로 분석합니다.

In [ ]:
convergence_rows = []
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    bands = list(context["bands"])
    trends_dir = Path(context["trends_dir"])
    convergence = plotConverge(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
        seed_bands=bands,
        include_corners=True,
        metric="best",
        threshold=0.5,
        dpi=300,
        show=False,
        save_path=trends_dir / "convergence.png",
    )
    convergence_gen = convergence["convergence"]["convergence_gen"]
    convergence_rows.append(
        {
            "algorithm": algorithm,
            "convergence_gen": convergence_gen,
        }
    )
    print(f"[{algorithm}] convergence generation: {convergence_gen}")

convergence_df = pd.DataFrame(convergence_rows).set_index("algorithm")
convergence_df


## 6. 전체 커버리지와 중첩 커버리지 분석

In [ ]:
report_rows = []
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    bands = list(context["bands"])
    report = saveReport(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
        output_dir=Path(context["report_dir"]),
        summary_dir=Path(context["final_dir"]),
        seed_bands=bands,
        include_corners=True,
        metric="best",
        threshold=0.5,
        target_values=TARGET_VALUES,
        dpi=300,
        show=False,
    )
    report_rows.append(
        {
            "algorithm": algorithm,
            **{
                key: report[key]
                for key in [
                    "convergence_plot",
                    "coverage_overlap_plot",
                    "coverage_overlap_summary",
                ]
            },
        }
    )

report_df = pd.DataFrame(report_rows).set_index("algorithm")
report_df


## 7. 논문 표용 요약 CSV

In [ ]:
summary_tables = {}
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    bands = list(context["bands"])
    final_dir = Path(context["final_dir"])
    summary = coverSummary(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
        seed_bands=bands,
        target_values=TARGET_VALUES,
    )
    summary_df = pd.DataFrame.from_dict(summary, orient="index")
    csv_path = final_dir / "summary.csv"
    summary_df.to_csv(csv_path, encoding="utf-8-sig")
    summary_tables[algorithm] = summary_df
    print(f"[{algorithm}] saved {csv_path}")

combined_summary = pd.concat(summary_tables, names=["algorithm", "group"])
combined_summary[
    [
        "runs",
        "n_sensors_mean",
        "coverage_percent_mean",
        "overlap_percent_of_target_mean",
        "overlap_percent_of_covered_mean",
        "redundant_hit_percent_mean",
        "logged_final_coverage_mean",
    ]
].round(3)
